[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/FFTjax/blob/main/notebooks/lin-elastic_gradient-check.ipynb)

# Forward-Mode vs. Reverse-Mode Gradients Through the Elastic Solver

Takes the exact same composite RVE and shear-strain setup as
[`lin-elastic_strain.ipynb`](./lin-elastic_strain.ipynb) and asks a different question: can we
differentiate the solve with respect to a material parameter (here, the fibre's Young's modulus
`E_fiber`), and do forward-mode (`jax.jvp`) and reverse-mode (`jax.grad`) autodiff agree on the
answer?

**Why this needs a different CG implementation than `solvers.mechanical.strain_nw_cg.solve_elastic`:**
the production solver uses `solvers.krylov.cg.cg_solve` internally (`jax.scipy.sparse.linalg.cg`, fast
early-exit via `jax.lax.while_loop`, adopted for speed). That was tested against this project's
FFT-based operators and found to give a **silently wrong** reverse-mode gradient (off by many
orders of magnitude on a heterogeneous composite, NaN on a degenerate homogeneous one) --
`while_loop` has no reverse-mode rule at all, and `jax.scipy.sparse.linalg.cg`'s own implicit-diff
workaround for that isn't reliable here. Forward-mode (`jax.jvp`) doesn't have this problem either
way.

So this notebook uses `solvers.krylov.cg.cg_solve_scan` instead: a fixed-length `jax.lax.scan` CG rather
than `jax.lax.while_loop` -- `scan`'s trip count is static, so it has a well-defined reverse-mode
rule. The cost is that it always runs the full step budget rather than stopping early. This
function started out defined locally in this notebook; it has since been promoted into
`solvers/krylov/cg.py` (validated there against this project's real, heterogeneous operators, not just
the synthetic check further down), so it's now imported rather than redefined below.

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running on Colab — installing FFTjax...")
    %pip install -q git+https://github.com/choROPeNt/FFTjax.git
else:
    import sys
    sys.path.insert(0, "../src")
    print("Running locally — using the local src/ checkout.")

import utils.precision  # side effect: configures JAX (X64 off on TPU, no GPU prealloc)
import jax
import jax.numpy as jnp
import numpy as np

from generation.rve import make_square_composite_rve
from operators.green import build_freq_grid, build_green_operator
from solvers.krylov.cg import cg_solve_scan

print("JAX backend:", jax.default_backend())
print("Devices:", jax.devices())
print("X64 enabled:", utils.precision.X64_ENABLED,
      "-> dtype:", jnp.zeros(1).dtype)

## Generate the composite RVE

Identical geometry to `lin-elastic_strain.ipynb`: a square-packed 2-fibre RVE, Vf~0.5, 5 um fibre
radius, 10-voxel-thick slab. This is fixed, non-differentiated data -- only the fibre's Young's
modulus below becomes a traced parameter.

In [2]:
phase_np, N, n, L, phi_act = make_square_composite_rve(
    phi=0.5, r_fiber=0.005, spacing=0.0002, N_min=32, nz=10,
)
Nv = int(np.prod(n))
dx = tuple(Li / ni for Li, ni in zip(L, n))
phase = jnp.array(phase_np.reshape(-1))   # 0 = matrix, 1 = fiber

xi_flat = build_freq_grid(n, L)

# Same prescribed macroscopic shear strain as lin-elastic_strain.ipynb.
eps_bar = jnp.array([
    [0.0, 1.0e-3, 0.0],
    [1.0e-3, 0.0, 0.0],
    [0.0, 0.0, 0.0],
])

print("grid n :", n)
print("fiber volume fraction (actual):", phi_act)

grid n : (89, 89, 10)
fiber volume fraction (actual): 0.5010730968312082


## A differentiable CG solve

`cg_solve_scan` (imported above from `solvers.krylov.cg`) is the same algorithm as `solve_elastic`'s
inner CG, just restructured around `jax.lax.scan` with a fixed `maxiter` instead of
`jax.lax.while_loop`, freezing the state once converged so the extra steps are no-ops (identical
solution to early-stopping CG, just not free).

`solve_elastic_diff(E_fiber)` rebuilds the matrix material's stiffness field, the reference-medium
Green's operator, and the Lippmann-Schwinger operator from scratch for a given `E_fiber`, then
solves with `cg_solve_scan` -- everything downstream of `E_fiber` is plain differentiable
`jax.numpy`, so both `jax.jvp` and `jax.grad` can trace all the way through.

In [3]:
def solve_elastic_diff(E_fiber, maxiter=100, toler_lin=1e-6):
    nu_matrix, nu_fiber = 0.35, 0.20
    E_matrix = 3.0e3

    def lame(E, nu):
        return E * nu / ((1 + nu) * (1 - 2 * nu)), E / (2 * (1 + nu))

    lam_m, mu_m = lame(E_matrix, nu_matrix)
    lam_f, mu_f = lame(E_fiber, nu_fiber)

    I2 = jnp.eye(3)
    def stiffness(lam, mu):
        return (lam * jnp.einsum('ij,kl->ijkl', I2, I2)
                + mu * (jnp.einsum('ik,jl->ijkl', I2, I2) + jnp.einsum('il,jk->ijkl', I2, I2)))

    C_matrix = stiffness(lam_m, mu_m)
    C_fiber = stiffness(lam_f, mu_f)
    C_field = ((1 - phase)[None, None, None, None, :] * C_matrix[..., None]
               + phase[None, None, None, None, :] * C_fiber[..., None])

    lam0 = 0.5 * (lam_m + lam_f)
    mu0 = 0.5 * (mu_m + mu_f)
    G_glob = build_green_operator(xi_flat, lam0, mu0, scheme="rotated", dx=dx)

    def fft_(x):
        s = x.shape
        return jnp.fft.fftn(x.reshape(s[:-1] + n), axes=(-3, -2, -1)).reshape(s)

    def ifft_(x):
        s = x.shape
        return jnp.fft.ifftn(x.reshape(s[:-1] + n), axes=(-3, -2, -1)).real.reshape(s)

    def A_op(v_flat):
        v = v_flat.reshape(3, 3, Nv)
        Cv = jnp.einsum("ijklm,klm->ijm", C_field, v)
        GCv = jnp.einsum("ijklm,klm->ijm", G_glob, fft_(Cv))
        return ifft_(GCv).reshape(-1)

    eps0 = jnp.ones((3, 3, Nv)) * eps_bar[:, :, None]
    sigma0 = jnp.einsum("ijklm,klm->ijm", C_field, eps0)
    bb = -ifft_(jnp.einsum("ijklm,klm->ijm", G_glob, fft_(sigma0))).reshape(-1)
    x0 = jnp.zeros_like(bb)

    delta_flat, converged = cg_solve_scan(A_op, bb, x0, toler_lin, maxiter)
    delta = delta_flat.reshape(3, 3, Nv)
    eps = eps0 + delta
    sigma = jnp.einsum("ijklm,klm->ijm", C_field, eps)
    return sigma, converged


def loss(E_fiber):
    """Macroscopic shear stress tau_xy as a function of the fibre's E -- the
    same quantity lin-elastic_strain.ipynb prints as `tau_xy (avg)`."""
    sigma, converged = solve_elastic_diff(E_fiber)
    return jnp.mean(sigma[1, 0])

## Compare forward-mode and reverse-mode gradients

`E_fiber = 70e3` MPa, matching the glass fibre in `lin-elastic_strain.ipynb`. There's no simple
closed-form gradient to check against here (unlike the homogeneous sanity checks elsewhere in this
project) -- the composite is heterogeneous, so the "proof" is forward-mode, reverse-mode, and a
central finite difference all agreeing with each other, independently computed three different
ways.

In [4]:
E_fiber0 = 70.0e3

# Reverse-mode: jax.grad (backpropagates through cg_solve_scan's jax.lax.scan).
loss_rev, grad_rev = jax.value_and_grad(loss)(E_fiber0)

# Forward-mode: jax.jvp (propagates a tangent through the same computation).
loss_fwd, grad_fwd = jax.jvp(loss, (E_fiber0,), (1.0,))

# Central finite difference, as a third, independent cross-check.
d_E = 1.0  # MPa
grad_fd = (loss(E_fiber0 + d_E) - loss(E_fiber0 - d_E)) / (2.0 * d_E)

print("loss (tau_xy) at E_fiber=70e3 MPa:", float(loss_rev))
print()
print("d(tau_xy)/d(E_fiber):")
print(f"  reverse-mode (jax.grad)      : {float(grad_rev):.10e}")
print(f"  forward-mode (jax.jvp)       : {float(grad_fwd):.10e}")
print(f"  central finite difference    : {float(grad_fd):.10e}")

rel_diff_rev_fwd = abs(float(grad_rev) - float(grad_fwd)) / abs(float(grad_fwd))
rel_diff_fd = abs(float(grad_fd) - float(grad_fwd)) / abs(float(grad_fwd))
print()
print(f"relative diff (reverse vs forward): {rel_diff_rev_fwd:.3e}")
print(f"relative diff (finite-diff vs forward): {rel_diff_fd:.3e}  (expect ~1e-4 to 1e-6, FD is only 1st-order accurate)")

assert rel_diff_rev_fwd < 1e-6, "forward- and reverse-mode gradients disagree!"
print("\nPASSED -- forward-mode and reverse-mode autodiff agree on d(tau_xy)/d(E_fiber).")

loss (tau_xy) at E_fiber=70e3 MPa: 7.625369073063829

d(tau_xy)/d(E_fiber):
  reverse-mode (jax.grad)      : 1.6463031759e-05
  forward-mode (jax.jvp)       : 1.6463031759e-05
  central finite difference    : 1.6463031758e-05

relative diff (reverse vs forward): 1.008e-14
relative diff (finite-diff vs forward): 9.893e-11  (expect ~1e-4 to 1e-6, FD is only 1st-order accurate)

PASSED -- forward-mode and reverse-mode autodiff agree on d(tau_xy)/d(E_fiber).


## Now check the production solver's backend directly

The claim above (`jax.scipy.sparse.linalg.cg`'s reverse-mode gradient is unreliable for this
project's operators) was stated, not shown. Let's actually check it here by calling the real
production solver, `solvers.mechanical.strain_nw_cg.dstrain_nw_cg` (aliased `solve_elastic`) --
same RVE, same `E_fiber`, same loss, only `C_field`/`G_glob` assembly duplicated since those come
from `E_fiber` itself. Its inner CG is `jax.scipy.sparse.linalg.cg` via `solvers.krylov.cg.cg_solve`,
not `cg_solve_scan`. Forward-mode should still agree with the `cg_solve_scan` result above;
reverse-mode is the one to watch.


In [ ]:
from solvers.mechanical.strain_nw_cg import dstrain_nw_cg


def solve_elastic_scipy_cg(E_fiber, maxiter=100, toler_lin=1e-6):
    """Same physics as solve_elastic_diff, but delegating the actual solve to
    the production solver (solvers.mechanical.strain_nw_cg.dstrain_nw_cg)
    instead of reimplementing the Lippmann-Schwinger operator by hand -- its
    inner CG is jax.scipy.sparse.linalg.cg (solvers.krylov.cg.cg_solve), not
    cg_solve_scan."""
    nu_matrix, nu_fiber = 0.35, 0.20
    E_matrix = 3.0e3

    def lame(E, nu):
        return E * nu / ((1 + nu) * (1 - 2 * nu)), E / (2 * (1 + nu))

    lam_m, mu_m = lame(E_matrix, nu_matrix)
    lam_f, mu_f = lame(E_fiber, nu_fiber)

    I2 = jnp.eye(3)
    def stiffness(lam, mu):
        return (lam * jnp.einsum('ij,kl->ijkl', I2, I2)
                + mu * (jnp.einsum('ik,jl->ijkl', I2, I2) + jnp.einsum('il,jk->ijkl', I2, I2)))

    C_matrix = stiffness(lam_m, mu_m)
    C_fiber = stiffness(lam_f, mu_f)
    C_field = ((1 - phase)[None, None, None, None, :] * C_matrix[..., None]
               + phase[None, None, None, None, :] * C_fiber[..., None])

    lam0 = 0.5 * (lam_m + lam_f)
    mu0 = 0.5 * (mu_m + mu_f)
    G_glob = build_green_operator(xi_flat, lam0, mu0, scheme="rotated", dx=dx)

    # The production solve, jax.scipy.sparse.linalg.cg backend and all --
    # early-exit via jax.lax.while_loop, same as solvers.mechanical.strain_nw_cg
    # uses for every solver in this project.
    _, sigma, _, _ = dstrain_nw_cg(n, C_field, G_glob, eps_bar, None, toler_lin, maxiter)
    return sigma


def loss_scipy_cg(E_fiber):
    sigma = solve_elastic_scipy_cg(E_fiber)
    return jnp.mean(sigma[1, 0])

In [6]:
loss_scipy_fwd, grad_scipy_fwd = jax.jvp(loss_scipy_cg, (E_fiber0,), (1.0,))
loss_scipy_rev, grad_scipy_rev = jax.value_and_grad(loss_scipy_cg)(E_fiber0)

print("jax.scipy.sparse.linalg.cg backend:")
print(f"  forward-mode (jax.jvp)  : {float(grad_scipy_fwd):.10e}")
print(f"  reverse-mode (jax.grad) : {float(grad_scipy_rev):.10e}")
print()
print("cg_solve_scan backend (trusted, matches finite difference above):")
print(f"  forward-mode (jax.jvp)  : {float(grad_fwd):.10e}")
print(f"  reverse-mode (jax.grad) : {float(grad_rev):.10e}")

fwd_rel_diff = abs(float(grad_scipy_fwd) - float(grad_fwd)) / abs(float(grad_fwd))
rev_rel_diff = abs(float(grad_scipy_rev) - float(grad_fwd)) / abs(float(grad_fwd))
print()
print(f"forward-mode relative diff vs. trusted value: {fwd_rel_diff:.3e}  (should be tiny)")
print(f"reverse-mode relative diff vs. trusted value: {rev_rel_diff:.3e}  (this is the bug)")

assert fwd_rel_diff < 1e-4, "forward-mode should agree regardless of CG backend"
# (tolerance here is 1e-4, not tighter, because the two CG backends each only
#  converge to their own toler_lin=1e-6 -- a few times that between two
#  *different* CG implementations reaching that tolerance is expected, not a bug)
print("\nConfirmed: jax.scipy.sparse.linalg.cg's forward-mode gradient is correct, but its\n"
      "reverse-mode gradient is not -- exactly why cg_solve_scan exists, and why the production\n"
      "solver (which uses jax.scipy.sparse.linalg.cg for speed) must not be wrapped in jax.grad.")

jax.scipy.sparse.linalg.cg backend:
  forward-mode (jax.jvp)  : 1.6463081136e-05
  reverse-mode (jax.grad) : 3.1692833757e+05

cg_solve_scan backend (trusted, matches finite difference above):
  forward-mode (jax.jvp)  : 1.6463031759e-05
  reverse-mode (jax.grad) : 1.6463031759e-05

forward-mode relative diff vs. trusted value: 2.999e-06  (should be tiny)
reverse-mode relative diff vs. trusted value: 1.925e+10  (this is the bug)

Confirmed: jax.scipy.sparse.linalg.cg's forward-mode gradient is correct, but its
reverse-mode gradient is not -- exactly why cg_solve_scan exists, and why the production
solver (which uses jax.scipy.sparse.linalg.cg for speed) must not be wrapped in jax.grad.


## Speed: what differentiability actually costs

Both solves below are just the forward pass (no `jax.grad`/`jax.jvp`), JIT-compiled and warmed up
first so compilation time doesn't pollute the timing -- same `maxiter=100` used throughout this
notebook, same composite RVE, same `E_fiber0`. This isolates the cost of `cg_solve_scan`'s
fixed-length `jax.lax.scan` (pays for all 100 steps, every call, which is what buys it a working
reverse-mode gradient) against `jax.scipy.sparse.linalg.cg`'s early-exit `jax.lax.while_loop`
(fast, but -- per above -- not safely differentiable here).

In [8]:
import time

solve_diff_jit = jax.jit(lambda E: solve_elastic_diff(E, maxiter=100))
solve_scipy_jit = jax.jit(lambda E: solve_elastic_scipy_cg(E, maxiter=100))

def bench(fn, repeats=10):
    out = jax.block_until_ready(fn(E_fiber0))   # warmup + compile, not timed
    t0 = time.perf_counter()
    for _ in range(repeats):
        out = jax.block_until_ready(fn(E_fiber0))
    return (time.perf_counter() - t0) / repeats * 1000, out

t_diff, out_diff = bench(solve_diff_jit)
t_scipy, out_scipy = bench(solve_scipy_jit)

sigma_diff = out_diff[0]
sigma_scipy = out_scipy
sol_rel_diff = float(jnp.max(jnp.abs(sigma_diff - sigma_scipy))) / float(jnp.max(jnp.abs(sigma_diff)))

print(f"cg_solve_scan (differentiable, scan, fixed maxiter=100): {t_diff:8.3f} ms/call")
print(f"jsla.cg       (production backend, while_loop, early-exit): {t_scipy:8.3f} ms/call")
print(f"speedup                                              : {t_diff / t_scipy:.2f}x")
print(f"solution relative difference                         : {sol_rel_diff:.2e}  (same physics either way)")

cg_solve_scan (differentiable, scan, fixed maxiter=100): 7653.303 ms/call
jsla.cg       (production backend, while_loop, early-exit): 2398.062 ms/call
speedup                                              : 3.19x
solution relative difference                         : 0.00e+00  (same physics either way)


## Next steps

- Swap `E_fiber` for any other material parameter (matrix `E`/`nu`, either phase's `nu`) -- nothing
  above is specific to the fibre modulus.
- This is the mechanism [`lin-elastic_strain.ipynb`](./lin-elastic_strain.ipynb)'s and
  [`lin-elastic_mixed-BC.ipynb`](./lin-elastic_mixed-BC.ipynb)'s solves would need to plug into
  for gradient-based inverse calibration (fit `E_fiber`, `E_matrix`, etc. to match a target
  effective stiffness or stress-strain curve via `optax`) -- `cg_solve_scan` (imported from
  `solvers.krylov.cg` above) is the missing piece. It has been promoted into the shared library, but
  isn't wired into `dstrain_nw_cg` itself: swapping the production solver's inner CG call from
  `cg_solve` to `cg_solve_scan` would make it always pay the fixed-`maxiter` cost noted above, so
  that's left as an explicit per-call choice for now rather than a default.
- For a heterogeneous problem at production scale, `cg_solve_scan`'s fixed `maxiter` needs to be
  tuned close to the problem's actual convergence (see the note on the
  [Benchmark](https://choROPeNt.github.io/FFTjax/documentation/benchmark) page) rather than left
  generous, since -- unlike `solve_elastic` -- every call here pays for the full budget.